# Aegis Health — `.litertlm` export (Google Colab, fresh runtime)

**Phase 5 only.** Pull merged FP16 checkpoint from HF Hub → quantize → bundle `.litertlm` → push to HF Hub. Must run in a **fresh Colab runtime** (`Runtime → Disconnect and delete runtime`) because `litert-torch-nightly` pulls `torch 2.11` and `transformers 5.6` which are incompatible with Unsloth. This is why SFT lives in a separate notebook.

**Runtime budget:** ~60–90 min (install 3 · download 10 · convert 30–60 · upload 15–25).

**Prerequisites:**
- `sft_train.ipynb` has completed and pushed the merged checkpoint to `V1rtucious/aegis-sft-e4b-merged-v2`
- Colab Secret `HF_TOKEN` set
- Empty private HF repo `V1rtucious/aegis-gemma-4-e4b-sft-litertlm-v2`
- Runtime: **Runtime → Change runtime type → CPU or GPU** (quantization is CPU-bound; GPU optional)

## Cell 1 — Nightly install

Install order matters. Dates pinned from HANDOVER.md §5.1; if a pinned nightly 404s, bump all three together — `!pip index versions litert-torch-nightly --pre | head -3` shows current.

**After this cell**, if the sanity probe at the bottom errors on import, `Runtime → Restart runtime` and re-run only this cell.

In [ ]:
# Force-reinstall ALL three nightlies. Without --force-reinstall on every line,
# pip silently keeps an existing older version it sees as "satisfying" the new
# pin, which is what bit us before — the bundle came out byte-identical to the
# 0.9.0.dev export because the old packages were still active.
!pip install --force-reinstall ai-edge-litert-nightly==2.2.0.dev20260428
!pip install --force-reinstall ai-edge-quantizer-nightly==0.7.0.dev20260429
!pip uninstall -y tensorflow
!pip install --force-reinstall litert-torch-nightly==0.10.0.dev20260429

# Why these specific versions (2026-04-30):
# litert-torch-nightly==0.10.0.dev* writes a v0.10 bundle ToC that
# litertlm-android:0.10.2 can parse. The earlier 0.9.0.dev* nightlies
# wrote a v0.9 bundle that the 0.10.x runtime mis-parses → SIGSEGV in
# liblitertlm_jni.so on first prefill (header diff verified by inspecting
# bytes 24-39 of the .litertlm: community bundle uses uint64 fields, the
# 0.9-exporter bundle has a uint32 field where uint64 is expected, shifting
# the flatbuffer start by 4 bytes).
#
# ai-edge-quantizer-nightly bumped from 0.6→0.7 around 2026-04-25 alongside
# the litert-torch major bump. The 0.6 quantizer's recipe registry differs
# from 0.7 — if "dynamic_wi4_afp32" stops resolving in Cell 4, check
# ai_edge_quantizer.recipe._RECIPE_REGISTRY for the new name.

# litert-torch-nightly pulls torch 2.11 + CUDA 13. Colab's preinstalled torchvision
# is torch-2.10+cu128 → CUDA-major mismatch breaks transformers' lazy import chain.
# We can't just remove torchvision: transformers 5.6.2 has Gemma3n which *requires*
# it during model-registry walks. Fix: install torchvision that matches torch 2.11+cu13.
!pip install --pre --force-reinstall torchvision --index-url https://download.pytorch.org/whl/nightly/cu130

print("\n" + "=" * 70)
print("REQUIRED: Runtime → Restart session, THEN run the sanity-probe cell below.")
print("If you reuse an old runtime where 0.9.0.dev was previously installed,")
print("--force-reinstall above will replace it; the sanity probe will confirm.")
print("=" * 70)

## Cell 1b — Sanity probe (run AFTER kernel restart)

After `Runtime → Restart session`, run ONLY this cell to verify imports. Skip Cell 1 — everything is already on disk.

In [ ]:
# Sanity probe — run after Runtime → Restart session.
# Hard-asserts the version pins so we catch a "pip kept the old version" bug
# BEFORE burning a 60-minute Cell 4 on the wrong exporter.

from importlib.metadata import version as _v
from litert_torch.generative.export_hf import export as _export_mod
from litert_torch.generative.export_hf.model_ext import gemma4 as _g4

import torch, transformers
print(f"torch              : {torch.__version__}   cuda={torch.cuda.is_available()}")
print(f"transformers       : {transformers.__version__}")
print(f"litert-torch       : {_v('litert-torch-nightly')}")
print(f"ai-edge-litert     : {_v('ai-edge-litert-nightly')}")
print(f"ai-edge-quantizer  : {_v('ai-edge-quantizer-nightly')}")
print(f"export module      : {_export_mod.__file__}")
print(f"gemma4 module      : {_g4.__file__}")

# Hard-fail if litert-torch is still on the 0.9 series — the bundle format
# is different and litertlm-android:0.10.2 can't read it.
_litert_ver = _v('litert-torch-nightly')
assert _litert_ver.startswith("0.10."), (
    f"\n\n!!! WRONG litert-torch VERSION: {_litert_ver}\n"
    f"Expected 0.10.x.dev*. The 0.9.x exporter writes bundles that\n"
    f"litertlm-android:0.10.2 cannot parse (SIGSEGV in prefill). Re-run\n"
    f"Cell 1 (it now uses --force-reinstall on all three packages),\n"
    f"then Runtime → Restart session, then re-run THIS cell.\n"
)
print("\nOK — exporter version aligns with Android runtime (0.10.x).")

## Cell 2 — Secrets + HF Hub login

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

## Cell 3 — Download merged FP16 checkpoint from HF Hub (~8 GB, 5–10 min with `hf_transfer`)

Pulls `V1rtucious/aegis-sft-e4b-merged-v2` (the FP16 merged weights pushed by `sft_train.ipynb`) into `/content/merged-v2/`. The export call in Cell 4 reads from this directory.

If you see a 404, your SFT notebook hasn't pushed yet — go check that first.

In [ ]:
from huggingface_hub import snapshot_download

MERGED_REPO = "V1rtucious/aegis-sft-e4b-merged-v4"

merged_path = snapshot_download(
    repo_id=MERGED_REPO,
    local_dir="/content/merged-v4",
    token=os.environ["HF_TOKEN"],
)
print(f"Merged checkpoint at: {merged_path}")
!du -sh /content/merged-v2
!ls /content/merged-v2 | head -20

## Cell 4 — Quantize + bundle `.litertlm`

**30–60 min, CPU-bound.** Reads `/content/merged-v2` from Cell 3, writes `/content/out/aegis_model.litertlm`.

Uses the production config: `dynamic_int4_asym` quantization, `cache_length=2048`, multi-resolution prefill `[128, 256, 512, 1024, 2048]`. Expected output: ~3.5 GB (community artifact = 3.65 GB).

**If this OOMs** (Colab Pro High-RAM ~51 GB can choke on the per_layer_embedder sub-export), uncomment the OOM-mitigated fallback inside the cell. That uses `single_token_embedder=True` + `experimental_use_mixed_precision=True` to skip the failing sub-export, but produces a smaller `cache_length=1024` model (~2.8-3.2 GB) — usable but with shorter effective context. Last-resort fallbacks (`dynamic_int4_sym` → `int4_blockwise` → unquantized) are documented in HANDOVER.md §5.3.

In [ ]:
from litert_torch.generative.export_hf.export import export

# === Production config (default) ===
# dynamic_wi4_afp32 = INT4 weight quantization, FP32 activations (weight-only quant).
# Other recipes like 'dynamic_int4_asym' or 'dynamic_int4_sym' are NOT registered
# in ai-edge-quantizer-nightly==0.6.0.dev20260424 — using them fails at the final
# quantize step after ~45 min of conversion. Stick with the wi4_afp32 naming.
# externalize_embedder=True is REQUIRED for Gemma 4 (the exporter asserts this).
#
# prefill_lengths: SINGLE bucket [1024] matches the litert-community Gemma 4 E4B
# bundle (verified working on litertlm-android:0.10.2). The earlier multi-resolution
# list [128, 256, 512, 1024, 2048] produced a 4.12 GB bundle that SIGSEGV'd on first
# prefill in liblitertlm_jni.so — the ToC for the 5 prefill graphs is read incorrectly
# by the 0.10.2 runtime against bundles produced by the 0.9.0-nightly exporter.
# Single bucket avoids that path; runtime chunks longer prompts into 1024-token prefills.
# Expected output: ~3.6 GB (matches community 3.65 GB).
export(
    model="/content/merged-v4",
    output_dir="/content/out",
    task="text_generation",
    bundle_litert_lm=True,
    quantization_recipe="dynamic_wi4_afp32",       # weight-only INT4 (known-good name)
    cache_length=2048,
    prefill_lengths=[1024],                        # single bucket; matches community
    externalize_embedder=True,                     # required for Gemma 4
)

# === OOM-mitigated fallback (only uncomment if the call above OOMs) ===
# Same quantization recipe, but with memory-pressure mitigations:
#   - cache_length=1024 (smaller KV cache during conversion)
#   - single_token_embedder=True (collapses per-layer embedder, skips OOMing sub-export)
#   - experimental_use_mixed_precision=True (reduces FP32 intermediates)
# Trade-off: shorter effective context window. Output ~2.8-3.2 GB.
#
# export(
#     model="/content/merged-v2",
#     output_dir="/content/out",
#     task="text_generation",
#     bundle_litert_lm=True,
#     quantization_recipe="dynamic_wi4_afp32",
#     cache_length=1024,
#     prefill_lengths=[1024],
#     externalize_embedder=True,                   # required for Gemma 4
#     single_token_embedder=True,                  # skip per-layer embedder sub-export
#     experimental_lightweight_conversion=True,
#     experimental_use_mixed_precision=True,       # shrink FP32 intermediates
# )

!ls -lh /content/out/

## Cell 5 — Upload `.litertlm` to HF Hub (~15–25 min for 3.5 GB)

In [ ]:
from huggingface_hub import HfApi, create_repo

LITERTLM_REPO = "V1rtucious/aegis-gemma-4-e4b-sft-litertlm-v4"
LOCAL_FILE = "/content/out/model.litertlm"        # what the exporter actually writes
REPO_FILE  = "aegis_model.litertlm"               # what we want it called on HF (matches HANDOVER.md)

create_repo(LITERTLM_REPO, private=True, exist_ok=True, token=os.environ["HF_TOKEN"])

# Upload ONLY the .litertlm file — not the whole /content/out/ folder, which also
# contains tmpXXXX/ scratch directories (~4 GB of intermediate .tflite files from
# the conversion pipeline that aren't needed at runtime).
HfApi().upload_file(
    path_or_fileobj=LOCAL_FILE,
    path_in_repo=REPO_FILE,
    repo_id=LITERTLM_REPO,
    repo_type="model",
    token=os.environ["HF_TOKEN"],
)
print(f"Published: https://huggingface.co/{LITERTLM_REPO}/blob/main/{REPO_FILE}")

## Cell 6 — Sanity-check the `.litertlm`

Cheap local check. Real validation is `adb push` + on-device functional test (HANDOVER.md §7).

In [ ]:
LOCAL_FILE = "/content/out/model.litertlm"        # exporter's actual output filename

with open(LOCAL_FILE, "rb") as f:
    head = f.read(16)
print("magic:", head[:8])  # expect b'LITERTLM'
print("hex  :", head.hex())

import os as _os
size_gb = _os.path.getsize(LOCAL_FILE) / (1024**3)
print(f"size : {size_gb:.2f} GB (community artifact = 3.65 GB; ours uses dynamic_wi4_afp32, expected ~3.5-4 GB)")